In [ ]:
import random
import glob, re, numpy as np
import matplotlib.pyplot as plt
import math
# collect files
files = []
for i in range(1, 100):
    files.extend(glob.glob(f"simulation-runs/500V_{i}.txt"))

# grab final (y,z) and final time from each file
y_last, z_last = [], []
t_last_ns_list = []  # drift times (ns)

for f in files:
    A = np.loadtxt(f, delimiter=",")
    y_last.append(A[-1, 3])     # y
    z_last.append(A[-1, 4])     # z
    t_last_ns_list.append(A[-1, 0])  # drift time (ns)

y_last = np.asarray(y_last)
z_last = np.asarray(z_last)

# --- minimal enclosing circle (Welzl) ---
def _dist(a, b): return np.hypot(a[0]-b[0], a[1]-b[1])
def _circle_from_2(a, b):
    c = ((a[0]+b[0])/2.0, (a[1]+b[1])/2.0); r = _dist(a, b)/2.0; return c, r
def _circle_from_3(a, b, c):
    ax, ay = a; bx, by = b; cx, cy = c
    d = 2*(ax*(by-cy) + bx*(cy-ay) + cx*(ay-by))
    if abs(d) < 1e-14:
        pairs = [(a,b), (a,c), (b,c)]
        c2, r2 = None, -1
        for u,v in pairs:
            cc, rr = _circle_from_2(u, v)
            if rr > r2 and all(_dist(w, cc) <= rr+1e-12 for w in (a,b,c)):
                c2, r2 = cc, rr
        return c2, r2
    ux = ((ax**2+ay**2)*(by-cy) + (bx**2+by**2)*(cy-ay) + (cx**2+cy**2)*(ay-by)) / d
    uy = ((ax**2+ay**2)*(cx-bx) + (bx**2+by**2)*(ax-cx) + (cx**2+cy**2)*(bx-ax)) / d
    center = (ux, uy); radius = max(_dist(center, a), _dist(center, b), _dist(center, c))
    return center, radius
def _contains(p, circle): c, r = circle; return _dist(p, c) <= r + 1e-12
def _mec_with_boundary(points, boundary):
    if not points or len(boundary) == 3:
        if len(boundary) == 0: return ((0.0, 0.0), 0.0)
        if len(boundary) == 1: return (boundary[0], 0.0)
        if len(boundary) == 2: return _circle_from_2(boundary[0], boundary[1])
        return _circle_from_3(boundary[0], boundary[1], boundary[2])
    p = points.pop()
    circle = _mec_with_boundary(points, boundary)
    if _contains(p, circle):
        points.append(p); return circle
    boundary.append(p)
    circle = _mec_with_boundary(points, boundary)
    boundary.pop(); points.append(p)
    return circle
def minimal_enclosing_circle(YZ):
    pts = [tuple(p) for p in np.asarray(YZ, dtype=float)]
    random.shuffle(pts)
    return _mec_with_boundary(pts, [])
# ----------------------------------------

pts = np.column_stack([y_last, z_last])
(center_y, center_z), radius_um = minimal_enclosing_circle(pts)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(y_last, z_last, s=16, c="tab:blue", alpha=0.7)

theta = np.linspace(0, 2*np.pi, 512)
cy = center_y + radius_um*np.cos(theta)
cz = center_z + radius_um*np.sin(theta)
ax.plot(cy, cz, linestyle=":", color="k", linewidth=2)
ax.scatter([center_y], [center_z], marker="x", s=80, color="red", zorder=3)

ax.annotate(
    f"R = {radius_um:.2f} µm",
    xy=(center_y + radius_um, center_z),
    xytext=(center_y + 1.2*radius_um, center_z + 0.15*radius_um),
    arrowprops=dict(arrowstyle="->", lw=1.2),
)
ax.text(center_y, center_z, f"({center_y:.2f}, {center_z:.2f}) µm", ha="left", va="bottom", fontsize=9)

ax.set_title("Final Electron Positions (y–z plane) with recombination$")
ax.set_xlabel("y (µm)"); ax.set_ylabel("z (µm)")
ax.grid(True, alpha=0.25)
ax.set_aspect("equal", adjustable="box")

# ---- NEW: average drift time label (bottom-left) ----
if len(t_last_ns_list):
    avg_t_ns = int(np.rint(np.mean(t_last_ns_list)))  # nearest integer
    ax.text(0.02, 0.02, f"Drift time ≈ {avg_t_ns} ns",
            transform=ax.transAxes, ha="left", va="bottom",
            fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.75, edgecolor="none"))

plt.tight_layout()
plt.show()

print(f"Circle center (y,z) = ({center_y: .3g}, {center_z:.3g}) µm;  radius = {radius_um:.3g} µm")
if len(t_last_ns_list):
    print(f"Average drift time ≈ {avg_t_ns} ns over {len(t_last_ns_list)} files")

In [ ]:

DATA_DIR = "simulation-runs"
pat = f"{DATA_DIR}/500V_*.txt"

TIME_COL, X_COL, Y_COL, Z_COL = 0, 1, 2, 3

files = []
for f in glob.glob(pat):
    m = re.search(r"500V_(\d+)\.txt$", f)
    if m: files.append((int(m.group(1)), f))
files.sort(key=lambda t: t[0])
if not files:
    raise RuntimeError(f"No files found matching {pat}")

t_last, x_last, y_last, z_last = [], [], [], []
for _, f in files:
    A = np.loadtxt(f, delimiter=",", usecols=(TIME_COL, X_COL, Y_COL, Z_COL))
    t_last.append(A[-1, 0])
    x_last.append(A[-1, 1])
    y_last.append(A[-1, 2])
    z_last.append(A[-1, 3])

t_last = np.asarray(t_last, float)
x_last = np.asarray(x_last, float)
y_last = np.asarray(y_last, float)
z_last = np.asarray(z_last, float)

def gaussian(x, mu, sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2) / (np.sqrt(2*np.pi)*sigma)

def plot_one(ax, data, axis_name):
    mu  = float(np.mean(data))
    sig = float(np.std(data, ddof=1))

    # show a wide window so the Gaussian is fully visible
    lo, hi = mu - 4*sig, mu + 4*sig
    lo = min(lo, np.min(data)); hi = max(hi, np.max(data))
    pad = 0.03*(hi - lo)
    lo -= pad; hi += pad

    # fixed, contiguous bins; stepfilled for a solid look
    nbins = 28
    bins = np.linspace(lo, hi, nbins+1)
    ax.hist(
        data, bins=bins, density=True,
        histtype="stepfilled", color="green",
        edgecolor="green", linewidth=0, alpha=0.75
    )

    xs = np.linspace(lo, hi, 1200)
    ax.plot(xs, gaussian(xs, mu, sig), "k--", lw=2)

    ax.set_xlim(lo, hi)
    ax.grid(True, alpha=0.25)
    ax.set_xlabel(f"{axis_name} spread [μm]")
    ax.text(0.98, 0.95,
            f"mean = {mu:.2f} μm\nσ = {sig:.2f} μm",
            ha="right", va="top", transform=ax.transAxes,
            bbox=dict(boxstyle="round", fc="white", ec="0.6", alpha=0.9))
    return mu, sig

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
fig.subplots_adjust(wspace=0.22)

mu_x, sig_x = plot_one(axes[0], x_last, "X")
mu_y, sig_y = plot_one(axes[1], y_last, "Y")
mu_z, sig_z = plot_one(axes[2], z_last, "Z")

axes[0].set_ylabel("Probability density")
fig.suptitle(
    f"E-field: 500 V/cm  (drift time ≈ {int(np.rint(np.mean(t_last)))} ns)",
    y=0.99
)
plt.tight_layout(rect=[0,0,1,0.95]) 
plt.show()

